<font size="5" color="red"><b>ch4. 머신러닝 모형 최적화</b></font>

# 1절. 변수 선택과 차원 축소## 1-1 변수선택과 차원축소
- 종속변수에 영향을 주는 변수들을 찾아 학습에 사용할 독립변수의 수를 줄임 (어떻게 하면 score를 높일 수 있을지?)
- 과적합과 변수들 사이의 다중공선성(변수들간 강한 상관관계)을 줄일 수 있음
 * 회귀계수 해석이 어려워짐. 모델 예측력이 좋아도 해석력이 떨어짐(어떤 변수가 제일 큰 요인인지 잘), p값이나 유의성 검정이 왜곡될 수 있음
- 모형의 학습 시간을 줄일 수 있음
- 주성분분석, 상관분석, **분류모형의 feature_importance_, 예측 모형의 coef_**
- SelectKBest : 가장 높은 score에 따라 K개의 특징을 선택

## 1-2 주성분분석(PCA, Principal Component Anaysis)
- 주성분분석은 변수 선택 및 차원축소 방법(기존의 모든 변수를 조합하여 새로운 변수로 만듦) 으로 널리 사용
- 주성분 분석은 상관관계가 있는 변수들을 선형결합해서 **분산이 극대화된 상관관계가 없는 새로운 변수(주성분)들로 축약**하는 것
- 주성분 분석은 사실 선형대수학이라기보다는 선형대수학의 활용적인 측면이 강하며 영상인식, 통계 데이터분석(주성분 찾기), 데이터 압축, 노이즈제거 등 여러 분야에 사용
- 영상처리에서 많이 활용 : 여러개의 영상 중 대표 이미지를 찾을 때 활용

In [ ]:
import seaborn as sns
from sklearn.decomposition import PCA
iris = sns.load_dataset('iris')
iris_X, iris_y = iris.iloc[:, :-1], iris.species
iris_X.head()

In [ ]:
pca = PCA(n_components=2) #n_components=2:주성분의 갯수
pca.fit(iris_X)
iris_pca = pca.transform(iris_X)
iris_pca[:5]

In [ ]:
# 각 주성분의 계수 : 각 주성분이 원래 특성들과 어떤 관계가 있는지 나타내는 가중치
pca.components_ #주성분 벡터
# 주성분1 = 0.36138659*x1 -0.08452251*x2 + 0.85667061*x3 + 0.3582892 *x4
# 주성분2 = -0.65658877*x1 + 0.73016143*x2 + 0.1733736*x3 + 0.07548102*x4

In [ ]:
# 설명분산 : 각 주성분이 원래 특성의 분산을 얼마나 설명하는지 나타내는 값
pca.explained_variance_

In [ ]:
# 설명분산율 : 각 주성분이 전체 분산에서 차지하는 비율(0~1 사이의 값) : 주성분 결과 특성은 97.76852%
pca.explained_variance_ratio_

## 1-3 상관관계 확인
- 각 변수들끼리의 상관관계 확인(시각화), 종속변수와 상관관계가 높은 변수들만 선택


In [ ]:
import pandas as pd
redwine = pd.read_csv('data/winequality-red.csv' , sep=';')
redwine.sample()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
# cmap의 종류 : https://jrc-park.tistory.com/155 
# http://seaborn.pydata.org/generated/seaborn.heatmap.html#seaborn.heatmap 
# http://seaborn.pydata.org/examples/many_pairwise_correlations.html 

In [ ]:
corr = redwine.corr()
# 상관관계 결과를 히트맵으로 시각화
# http://seaborn.pydata.org/examples/many_pairwise_correlations.html
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={"shrink": .8})


In [ ]:
plt.figure(figsize=(16, 8))
sns.heatmap(corr, annot=True, fmt='.3f', vmin=-1, vmax=1, cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
plt.show()

In [ ]:
np.triu(np.ones_like(corr), k=0) # 대각선 포함 위가 1인 삼각행렬
np.triu(np.ones_like(corr), k=1) # 대각선 제외 위가 1인 삼각행렬
np.tril(np.ones_like(corr), k=0) # 대각선 포함 아래가 1인 삼각행렬
np.tril(np.ones_like(corr), k=-1) # 대각선 제외 아래가 1인 삼각행렬

In [ ]:
plt.figure(figsize=(16,6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', vmin=-1, vmax=1, cmap='coolwarm_r', mask=mask)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

## 1-4 분류모형의 Feature importance
- 분류모형의 feature_importance_  속성은 각 독립변수들이 종속변수에 영향을 주는 정도
- LogisticRegression이나 SVC, MLP, GaussianNB등은 feature_importance_가 없음
- 그 외 분류모형은 사용가능

In [ ]:
from sklearn.model_selection import train_test_split
X = redwine.iloc[:, :-1]  # 마지막 열을 제외한 모든 열
y = redwine.iloc[:, -1]   # 마지막 열만 선택
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3)
train_X.shape, test_X.shape, train_y.shape, test_y.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=10, #트리 갯수 10개
                                  random_state=10, #랜덤시드 고정
                                  )
rf_model.fit(train_X, train_y)

In [ ]:
rf_model.feature_importances_ # 특성 중요도

In [ ]:
train_X.columns

In [ ]:
features = pd.DataFrame(data =  np.c_[X.columns, rf_model.feature_importances_], # 특성 중요도와 특성 이름을 합쳐서 출력
                        columns=['feature', 'importance'])
features['importance'].sum() 

In [ ]:
features.sort_values(by='importance', ascending=False, inplace=True)
features.reset_index(drop=True, inplace=True)
features

### feature_importance_를 이용한 변수 중요도 시각화

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(features['feature'], features['importance'])
plt.xticks(rotation=45, fontsize=10)
plt.show()

In [ ]:
features.importance

In [ ]:
# features.importance 누적합
# l = [1, 2, 3]
# np.cumsum(l)
y_stack = np.cumsum(features.importance)
np.c_[features.importance, y_stack]

In [ ]:
# 누적합을 이용해 시각화
plt.figure(figsize=(12, 4))
plt.bar(features['feature'], y_stack)
plt.plot(features['feature'], y_stack, color='red', marker='o', lw=3)
plt.xticks(rotation=45, fontsize=10)
plt.show()

### RFE(Recursive Feature Elimination) 방식
- RFE 클래스를 이용 : 중요도에 따라 중요도가 낮은 변수부터 하나씩 제거해 가면서 최종 선택된 변수 개수만큼 중요도가 높은 변수를 찾는다.

In [ ]:
# 5개 특징이 남을 때까지 변수를 제거(기준: feature_importances_)
from sklearn.feature_selection import RFE
rfe = RFE(estimator=rf_model, n_features_to_select=5)
rfe.fit(train_X, train_y)
rfe.get_support() # 선택된 특성의 인덱스 반환

In [ ]:
features_rfe = pd.DataFrame(data =  np.c_[X.columns, rfe.get_support()],
                            columns=['feature', 'selected'])
features_rfe[features_rfe['selected']] # 선택된 특성만 출력

## 1-5 SelectKBest
- 가장 높은 score에 따라 k개 feature 선택

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, chi2
X = iris.iloc[:, :-1].values
y = iris.iloc[:, -1].values
X.shape, y.shape, type(X), type(y) 

In [ ]:
feature_names = iris.columns[:-1].tolist()
feature_names

In [ ]:
# 가장 중요한 feature 1개 추출
# f_classif : y는 범주형. x는 연속형
# chi2 : y는 범주형. x는 범주형, 양의 실수
# mutual_info : 비선형 데이터 고려...
X_new = SelectKBest(f_classif, # x가 연속형
                    k=1 # 추출될 feature의 갯수
                    ).fit_transform(X, y) # X와 y를 이용해 fit하고 변환
X_new.shape, type(X_new)

# 2절. 파라미터 선택
- 하이퍼파라미터(사용자가 직접 설정할 수 있는 파라미터). 최적의 결과를 내는 하이퍼파라미터값?
    1. validation_curve() : 단일 하이퍼 파라미터 최적화 함수
    2. GridSearchCV: 그리드를 사용한 복수 하이퍼 파라미터 최적화 클래스(가장 높은 score를 내는 모형까지 찾아줌)

## 2-1 validation_curve()
- param_name, param_range(리스트), scoring(성능기준지표) 매개변수로 받아 최적의 성능 계산

In [ ]:
# 데이터
from sklearn.datasets import load_digits
digits = load_digits()
# digits.data : (1797, 64) # 1797개의 샘플, 64개의 특성
# digits.image.shape : (1797, 8, 8) # 1797개의 샘플, 8x8 이미지
# digits.target : (1797,) 열 배열
# digits.target_names : 타겟변수 내용
X, y = digits.data, digits.target
X.shape, y.shape 

In [ ]:
X[0].reshape(8, 8) == digits.images[0] # 8x8 이미지로 변환

In [ ]:
plt.figure(figsize=(1,1))
plt.imshow(X[0].reshape(8, 8), cmap='gray_r')
plt.title(y[0])
plt.axis('off')  # 축 제거
plt.show()


In [ ]:
from sklearn.svm import SVC
model = SVC(probability=True) # 확률 예측을 위해 probability=True 설정
model.fit(X, y)

In [ ]:
model.predict(X[0].reshape(1, -1)) # 0번째 샘플 예측


In [ ]:
model.predict(X[:5]) # 0~4번째 샘플 예측

In [ ]:
# 예측확률
print(model.classes_)
print(model.predict_proba(X[0].reshape(1, -1))) # 0번째 샘플 예측 확률

In [ ]:
for c, p in zip(model.classes_, model.predict_proba(X[0].reshape(1, -1))[0]):
    print(f"{c} : {p:.3f}") # 각 클래스의 예측 확률 출력

In [ ]:
model.score(X, y)

In [ ]:
range = np.array([0, 1, 2, 3])
score = np.array([0.5, 0.6, 0.65, 0.5])
plt.figure(figsize=(4,2))
plt.semilogx(range, score, color='blue', marker='o', lw=2)
plt.fill_between(range, score-0.1, score+0.1, color='blue', alpha=0.2)
plt.show()

In [ ]:
# SVC() 도형에서 C 파라미터 값을 다음의 범위 중 제일 좋은 C값?
# 10의 -6승부터 10의 -1승까지 로그간격으로 균등분포 10개를 추출
param_range = np.logspace(-6, -1, 10)
param_range

In [ ]:
%%time
from sklearn.model_selection import validation_curve
train_score, test_score = validation_curve(
    SVC(), # 예측모형
    X, y,
    param_name="gamma",
    param_range=param_range, # list로 파라미터 전달
    cv=10, # 교차검증 : 데이터 10개중 1개씩 test 데이터로 검증하고 평균 score
    scoring="accuracy",
    n_jobs=-1 # 시스템의 모든 core 사용
)

In [ ]:
test_score.shape, train_score.shape

In [ ]:
train_score_mean = np.mean(train_score, axis=1) # 행별 평균
test_score_mean = np.mean(test_score, axis=1)
train_score_std = np.std(train_score, axis=1)
test_score_std = np.std(test_score, axis=1)

In [ ]:
train_score_mean

In [ ]:
test_score_mean

In [ ]:
plt.figure(figsize=(8,5))
plt.semilogx(param_range, train_score_mean)
plt.fill_between(param_range,
                 train_score_mean - train_score_std,
                 train_score_mean + train_score_std,
                 color='blue', alpha=0.2)

In [ ]:
plt.figure(figsize=(8,5))
plt.semilogx(param_range, train_score_mean)
plt.fill_between(param_range, train_score_mean-train_score_std,
                train_score_mean+train_score_std,
                alpha=0.2,
                color='blue')
plt.semilogx(param_range, test_score_mean)
plt.fill_between(param_range, test_score_mean-test_score_std,
                test_score_mean+test_score_std,
                alpha=0.2,
                color='red')
plt.scatter(param_range, test_score_mean, c='k')

In [ ]:
test_score_mean.argmax()

In [ ]:
# 최적의 gamma 값
gammar = param_range[6]
gammar

In [ ]:
model = SVC(gamma=gammar).fit(X, y) # 최적의 gamma 값으로 모델 학습

In [ ]:
model.score(X, y) # 모델 평가

## 2-2 GridSearchCV
- 복수개의 하이퍼 파라미터 최적화 클래스
- 모형도 가지고 옴
- fit(), score(), predict(), predict_proba(), decision_funcion()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest
from sklearn.svm import SVC

# 데이터
import pandas as pd
redwine = pd.read_csv('data/winequality-red.csv' , sep=';')
redwine_X, redwine_y = redwine.iloc[:, :-1], redwine.iloc[:, -1]
redwine_X.shape, redwine_y.shape

In [ ]:
%%time
# SelectKBest로 최적의 독립변수 k개 찾음 -> SVC()에서 최적의 C값을 찾음
selection = SelectKBest(k=1) # 가장 평가 점수가 높은 k개 찾음
svc = SVC(kernel='linear') # 직선으로 나누는 가장 단순한 분류모형
pipeline = Pipeline([('select', selection), ('svc', svc)])
param_grid = dict(select__k = [4,5,6,7,8,9,10,11],
        svc__C = [0.1, 1, 10] # 오차 범위 허용 정도(큰C:엄격, 작은C:허용범위 큼)
)
grid_search = GridSearchCV(
                pipeline,
                param_grid=param_grid, # 파라미터들
                cv=2,
                verbose=2, # 로그 출력의 수다스러운 정도
                n_jobs=-1
)
grid_search.fit(redwine_X.values, redwine_y.values)

In [ ]:
# 최적의 파라미터값
grid_search.best_params_

In [ ]:
# 최적의 모형
model = grid_search.best_estimator_
model.score(redwine_X.values, redwine_y.values)

In [ ]:
model = SVC().fit(redwine_X.values, redwine_y.values)
model.score(redwine_X.values, redwine_y.values)

# 3절. 자료 불균형 처리
- 단순 언더/오버 샘플링
- 단, 단순 오버샘플링시 소수의 데이터를 복사하면, 과적합 우려
- 오버샘플링하는 방법 : SMOTE
## 3-1 SMOTE를 이용한 오버샘플링 전

In [ ]:
# 데이터
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=10000,
                          n_features=10, # 독립변수 갯수
                          n_informative=5, # 타겟변수에 영향을 미치는 독립변수
                          n_redundant=0,
                          n_clusters_per_class=1,
                          n_classes=2,
                          weights=[0.99, 0.01], # 각 클래스에 할당된 표본 비율
                          random_state=42
)
y.mean()

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', alpha=0.4)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                   test_size=0.3,
                                                   stratify=y,
                                                   random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100,
                                 max_features=2,
                                 random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
y_hat = rf_model.predict(X_test)
from sklearn.metrics import confusion_matrix, classification_report
confusion_matrix(y_test, y_hat)

In [ ]:
print(classification_report(y_test, y_hat))

## 3-2 SMOTE를 이용한 전체 오버샘플링 후 데이터 셋 분리
- imbalanced-learn 라이브러리 install

In [ ]:
# 0그룹과 1그룹의 갯수
df = pd.DataFrame(np.c_[X, y])
df.iloc[:,-1].value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE() # 0그룹 : 1그룹 = 1:1
# sm = SMOTE(sampling_strategy={0:9860, 1:420})
X_resampled, y_resample = sm.fit_resample(X, y)
X_resampled.shape, y_resample.shape

In [ ]:
# 0그룹과 1그룹의 갯수
df = pd.DataFrame(np.c_[X_resampled, y_resample])
df.iloc[:,-1].value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resample,
                                                   test_size=0.3,
                                                   stratify=y_resample,
                                                   random_state=42)
rf_model = RandomForestClassifier(n_estimators=100,
                                 max_features=2,
                                 random_state=42)
rf_model.fit(X_train, y_train)
y_hat = rf_model.predict(X_test)
confusion_matrix(y_test, y_hat)

In [ ]:
print(classification_report(y_test, y_hat))

## 3-3 가중치 제어
- 자료 불균형 처리의 또 다른 방법
- sklearn의 예측 모형에서 class_weigh 매개변수 설정

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100,
                                 max_features=2,
                                 class_weight={0:1, 1:1.4},
                                 random_state=42)
rf_model.fit(X_train, y_train)

# 4절. 앙상블 모형
- 목적 : 여러 분류모형을 하나의 메타 분류모델로 연결하여 개별 모형보다 더 좋은 일반화 성을 달성
- 방법 : 
    * 하나의 메타 분류 알고리즘 이용 : 배깅(bagging), 부스팅(boosting)
    * 여러 분류 알고리즘을 이용 : 다수결투표
- 배길 vs 부스팅
    * 배깅
        - 복원추출로 데이터를 뽑아 병렬 학습 후 score가 높은 모델에 가중치 부여
        - 과적합 줄일 수 있음
        - 데이터가 충분하고 과적합을 방지하면서 안정적인 모델이 필요할 때
        - RandomForestClassifier, BaggingClassifier
    * 부스팅
        - 순차적으로 모델 학습. 앞의 모델에서 틀린 데이터의 50%를 재학습
        - 오답에 가중치를 둠(오답에 더 집중)
        - 성능이 극대화
        - 성능 극대화해야 하는데, 데이터가 비교적 적거나 복잡한 패턴의 학습을 해야할 경우

## 4-1 배깅알고리즘

In [ ]:
wine_df = pd.read_csv('data/wine.csv')
wine_df.head()

In [ ]:
X = wine_df.iloc[:, 1:]
y = wine_df['Class label']
X.shape, y.shape

In [ ]:
y.value_counts()

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(X, y,
                                                   test_size=0.3,
                                                   stratify=y,
                                                   random_state=1)

In [ ]:
# 의사결정나무 알고리즘
from sklearn.tree import DecisionTreeClassifier
tree_model = DecisionTreeClassifier(criterion='entropy', random_state=1)
tree_model.fit(train_X, train_y)
tree_model.score(test_X, test_y)

In [ ]:
# 배깅알고리즘
from sklearn.ensemble import BaggingClassifier
bag_model = BaggingClassifier(base_estimator=tree_model,
                              n_estimators=500, # 트리 갯수
                              bootstrap=True, # 부트스트랩 샘플링
                              bootstrap_features= False, # 모든 특성을 사용
                            #   n_jobs=-1, # 모든 코어 사용
                              random_state=1) # 랜덤시드 고정
bag_model.fit(train_X, train_y)
bag_model.score(test_X, test_y)

In [ ]:
# 랜덤포레스트 알고리즘
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier().fit(train_X, train_y)
rf_model.score(test_X, test_y)

### 배깅알고리즘시 0.632 규칙

In [ ]:
np.random.choice(10, 10, replace=True) # 0~9까지의 숫자 중에서 10개를 복원추출 뽑기

In [ ]:
len(set(np.random.choice(10000, 10000, replace=True))) # 중복제거

In [ ]:
N = 10000000
len(set(np.random.choice(N, N))) / N # 0.632 규칙

## 4-2 임의의 데이터를 만들어 최적 모형 탐색

In [ ]:
X, y = make_classification(n_samples=1000,
                          n_features=10,
                          n_informative=5,
                          n_redundant=0,
                          n_classes=2,
                          n_clusters_per_class=1,
                          weights=[0.9, 0.1], # 0그룹인 90%, 1그룹10%
                          random_state=42)

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(X, y,
                                                   test_size=0.3,
                                                   stratify=y,
                                                   random_state=42)
sm = SMOTE() # 0그룹 : 1그룹 = 1:1
resampled_X, resampled_y = sm.fit_resample(train_X, train_y)
resampled_X.shape, resampled_y.shape, test_X.shape, test_y.shape

In [ ]:
def model_measure(model, train_X=resampled_X, train_y=resampled_y,
                    test_X=test_X, test_y=test_y):
        '매개변수로 들어온 model 학습 후 accuracy, precision, recall, f1-score를 반환하는 함수'
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
        model.fit(train_X, train_y)
        y_hat = model.predict(test_X) # 모델 예측값
        accuracy = model.score(test_X, test_y) # 정확도
        precision = precision_score(test_y, y_hat) # 정밀도
        recall = recall_score(test_y, y_hat) # 재현율
        f1score = f1_score(test_y, y_hat) # F1-score
        return "정확도 : {:.3f}, 정밀도 : {:.3f}, 재현율 : {:.3f}, F1-score : {:.3f}".format(
            accuracy, precision, recall, f1score
        )

In [ ]:
model_measure(RandomForestClassifier(n_estimators=100,
                                     random_state=42,
                                     max_features=2))

In [ ]:
model_measure(SVC(random_state=42))

## 4-3 부스팅 알고리즘
- Adaboost, XGB, LGBM, CatBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
model_measure(AdaBoostClassifier())

In [ ]:
from xgboost import XGBClassifier
model_measure(XGBClassifier(max_depth=10, # 트리의 깊이
                           n_estimators=100, # 트리의 갯수,
                           learning_rate=0.01, # 학습률
                           ))

In [ ]:
from lightgbm import LGBMClassifier # pip install lightgbm
model_measure(LGBMClassifier(force_col_wise=True, # 특성별로 병렬처리
                           n_estimators=100, # 트리의 갯수 
                           ))

## 4-4 투표를 이용한 앙상블
- voting='hard' : 다수결로 투표
- voting='soft' : 확률의 합을 계싼한 투표

In [ ]:
X, y = make_classification(n_samples=200,
                          n_features=2,
                          n_informative=2,
                          n_redundant=0,
                          n_classes=2,
                          n_clusters_per_class=1,
                          random_state=42)
train_X, test_X, train_y, test_y = train_test_split(X, y,
                                                   test_size=0.3,
                                                   stratify=y,
                                                   random_state=42)

In [ ]:
tf_model = RandomForestClassifier(max_features=2, random_state=42)
xgb_model = XGBClassifier(max_depth=10, # 트리의 깊이
                           n_estimators=100, # 트리의 갯수,
                            learning_rate=0.01, # 학습률
                            eval_metric='logloss', # 평가 지표
)
lgb_model = LGBMClassifier(force_col_wise=True, # 특성별로 병렬처리
                           n_estimators=100, # 트리의 갯수
                           verbose=-1, # 로그 출력 안함
                           )
print(model_measure(tf_model))
print(model_measure(xgb_model))
print(model_measure(lgb_model))

In [ ]:
from sklearn.ensemble import VotingClassifier
v_model = VotingClassifier(estimators=[('rfm',rf_model),
                                       ('xgb',xgb_model),
                                       ('lgb',lgb_model)],
                            voting='hard', # 다수결 투표
                            )
model_measure(v_model) # 측정